# Vector Auto Regression Hands-On

In [1]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Import important libraries
import pandas as pd
import yfinance as yf
import datetime as dt
from nsepy import get_history as gh
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_percentage_error as mae
from statsmodels.tsa.api import VAR
from stockFunctions import rmsemape
from stockFunctions import graph
from stockFunctions import conversionSingle

In [3]:
# Dataset collection
stock_symbol = "RELIANCE.NS"
Stk_data =yf.download(stock_symbol, start = "2023-01-01", end = "2023-07-01")
Stk_data.info() # Check the quality of dataset

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 122 entries, 2023-01-02 to 2023-06-30
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Close, RELIANCE.NS)   122 non-null    float64
 1   (High, RELIANCE.NS)    122 non-null    float64
 2   (Low, RELIANCE.NS)     122 non-null    float64
 3   (Open, RELIANCE.NS)    122 non-null    float64
 4   (Volume, RELIANCE.NS)  122 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 5.7 KB


In [4]:
# Check parameters and values in the dataset
Stk_data

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2023-01-02,1170.477783,1171.886424,1157.891000,1158.708901,5316175
2023-01-03,1161.912354,1169.160003,1157.709267,1165.547536,7658932
2023-01-04,1144.418213,1163.730055,1142.350684,1161.889687,9264891
2023-01-05,1142.373535,1152.529332,1137.806870,1146.667607,13637099
2023-01-06,1152.756348,1157.777455,1144.304622,1148.098818,6349597
...,...,...,...,...,...
2023-06-23,1142.691528,1151.165970,1141.441900,1149.121157,6628570
2023-06-26,1133.967163,1142.986912,1130.854571,1139.170000,12641159


In [5]:
# Consideration of important parameters of the dataset
Stk_data = Stk_data[["Close", "High", "Low", "Open"]]
Stk_data

Price,Close,High,Low,Open
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,
2023-01-02,1170.477783,1171.886424,1157.891000,1158.708901
2023-01-03,1161.912354,1169.160003,1157.709267,1165.547536
2023-01-04,1144.418213,1163.730055,1142.350684,1161.889687
2023-01-05,1142.373535,1152.529332,1137.806870,1146.667607
2023-01-06,1152.756348,1157.777455,1144.304622,1148.098818
...,...,...,...,...
2023-06-23,1142.691528,1151.165970,1141.441900,1149.121157
2023-06-26,1133.967163,1142.986912,1130.854571,1139.170000


In [6]:
# Separate the functions to carry out univariate analysis
Columns = Stk_data[["Close", "High", "Low", "Open"]]
Columns

Price,Close,High,Low,Open
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,
2023-01-02,1170.477783,1171.886424,1157.891000,1158.708901
2023-01-03,1161.912354,1169.160003,1157.709267,1165.547536
2023-01-04,1144.418213,1163.730055,1142.350684,1161.889687
2023-01-05,1142.373535,1152.529332,1137.806870,1146.667607
2023-01-06,1152.756348,1157.777455,1144.304622,1148.098818
...,...,...,...,...
2023-06-23,1142.691528,1151.165970,1141.441900,1149.121157
2023-06-26,1133.967163,1142.986912,1130.854571,1139.170000


In [7]:
# Model creation
MMS = MinMaxScaler()
Data1 = MMS.fit_transform(Columns)
print("Length of data is:", Data1.shape)

Length of data is: (122, 4)


In [8]:
Data1 = pd.DataFrame(Data1, columns = ["Close", "High", "Low", "Open"])

In [9]:
# Training of model
Training_size = round(len(Data1)*0.8) # Here 0.8 is a playable parameter. The developer can use different values to get an optimised output.
print(Training_size)
X_train = Data1[:Training_size]
X_test = Data1[Training_size:]
print (f"The length of X_train is: {X_train.shape}")
print (f"The length of X_test is: {X_test.shape}")
# Note: In the case of TSA, the input and output are the same. This is because TSA is univariate analysis
Y_train = X_train
Y_test = X_test

98
The length of X_train is: (98, 4)
The length of X_test is: (24, 4)


In [10]:
Performance = {"Model":[], "RMSE":[], "MAPE":[], "Lag":[], "Test":[]}

In [11]:
def combination (dataset, list):
    print (list)
    dataset2 = dataset[list]
    Test_obs = 22
    Train = dataset2[:-Test_obs]
    Test = dataset2[-Test_obs:]
    for i in [1,2,3,4,5,6,7,8,9,10]:
        Model = VAR(Train)
        Results = Model.fit(i)
        print ("Order =", i)
        print ("AIC =", Results.aic)
        print ("BIC =", Results.bic)
        print()
    X = Model.select_order(maxlags = 12)
    Order = X.selected_orders["aic"]
    Result = Model.fit(Order)
    Lagged_values = Train.values[-Order:]
    Pred = Result.forecast(y = Lagged_values, steps = Test_obs)
    Prediction = pd.DataFrame(Pred, columns = list)
    Prediction.to_csv("VARForecasted_{}.csv".format(Test_obs))
    RMSE = round(mse(Test, Pred, squared = False))
    MAPE = mae(Test, Pred)
    Performance["Model"].append(list)
    Performance["RMSE"].append(RMSE)
    Performance["MAPE"].append(MAPE)
    Performance["Lag"].append(Order)
    Performance["Test"].append(Test_obs)
    Perf = pd.DataFrame(Performance)
    return Perf, Result, Pred

In [12]:
list = ["Close", "High", "Low", "Open"]

In [13]:
Perf, Result, Pred = combination (Data1, list)

['Close', 'High', 'Low', 'Open']
Order = 1
AIC = -27.54791083326121
BIC = -27.023644196870382

Order = 2
AIC = -27.47558014881893
BIC = -26.52600025869505

Order = 3
AIC = -27.40582920506591
BIC = -26.0255717732909

Order = 4
AIC = -27.183541632958182
BIC = -25.367128330668464

Order = 5
AIC = -27.030704855948063
BIC = -24.772540025480215

Order = 6
AIC = -27.033296371441324
BIC = -24.327663624345572

Order = 7
AIC = -27.03140988575196
BIC = -23.872468582464027

Order = 8
AIC = -26.95240291872896
BIC = -23.334184525571644

Order = 9
AIC = -26.806363323936846
BIC = -22.722767643008343

Order = 10
AIC = -26.920473238815724
BIC = -22.36526450621391



In [14]:
Data1

,Close,High,Low,Open
0,0.947162,0.930052,0.968437,0.856777
1,0.899507,0.914507,0.967385,0.895268
2,0.802174,0.883549,0.878484,0.874680
3,0.790798,0.819690,0.852183,0.789003
4,0.848565,0.849611,0.889794,0.797059
...,...,...,...,...
117,0.792568,0.811917,0.873224,0.802813
118,0.744028,0.765285,0.811941,0.746803
119,0.746304,0.750000,0.806550,0.720077
120,0.829858,0.823446,0.836007,0.762149


In [15]:
Perf

,Model,RMSE,MAPE,Lag,Test
0,"[Close, High, Low, Open]",0,0.236499,1,22
